In [1]:
# STEP 1: AUTOMATIC LIBRARY INSTALLATION
import os
import sys

print("Checking and installing required libraries. Please wait...")
os.system(f"{sys.executable} -m pip install -q gradio deep-translator gtts")
print("Libraries installed successfully!")

# STEP 2: MAIN TRANSLATION & UI CODE
import gradio as gr
from deep_translator import GoogleTranslator
from gtts import gTTS
import gtts.lang

# 1. Fetch ALL text translation languages available in the world from Google Translate
ALL_WORLD_TEXT_LANGS = {lang.title(): code for lang, code in GoogleTranslator().get_supported_languages(as_dict=True).items()}

# 2. Fetch native voice capabilities from gTTS to check for speech availability
NATIVE_VOICE_LANGS = gtts.lang.tts_langs()

# 3. Comprehensive Mapping for regional/minority languages lacking an official gTTS voice.
# This ensures that languages like Bhojpuri or Sanskrit can still be read out loud using matching accents.
REGIONAL_SPEECH_FALLBACKS = {
    "bhojpuri": "hi",    # Fallback Bhojpuri text to Hindi voice
    "maithili": "hi",    # Fallback Maithili text to Hindi voice
    "dogri": "hi",       # Fallback Dogri text to Hindi voice
    "sanskrit": "hi",    # Fallback Sanskrit text to Hindi voice
    "awadhi": "hi",      # Fallback Awadhi text to Hindi voice
    "chhattisgarhi": "hi", # Fallback Chhattisgarhi text to Hindi voice
    "haryana": "hi",     # Fallback Haryanvi text to Hindi voice
    "konkani": "mr",     # Fallback Konkani text to Marathi voice
    "punjabi": "pa",     # Route text explicitly to Punjabi voice code
    "marwari": "hi",     # Fallback Marwari text to Hindi voice
}

# 4. Initialize Source Language Dropdown with Auto-Detect option
SOURCE_LANGUAGES = {"Auto Detect": "auto"}
SOURCE_LANGUAGES.update(ALL_WORLD_TEXT_LANGS)


def translate_and_tts(text, source_lang, target_lang):
    """
    Main function to handle translation for ALL world languages with stable audio playback.
    """
    if not text.strip():
        return "Please enter some text to translate.", None

    try:
        # Get the corresponding language translation codes
        src_code = SOURCE_LANGUAGES[source_lang]
        tgt_code = ALL_WORLD_TEXT_LANGS[target_lang]

        # Initialize the translator and process the translation
        translator = GoogleTranslator(source=src_code, target=tgt_code)
        translated_text = translator.translate(text)

        # Smart Audio Routing Engine
        voice_code = tgt_code
        using_fallback = False
        fallback_name = ""

        # If the target language code is not natively supported by the voice engine
        if voice_code not in NATIVE_VOICE_LANGS:
            # Check our custom regional map
            fallback_code = REGIONAL_SPEECH_FALLBACKS.get(target_lang.lower())
            if fallback_code:
                voice_code = fallback_code
                using_fallback = True
                fallback_name = [name for name, code in ALL_WORLD_TEXT_LANGS.items() if code == fallback_code][0]
            else:
                # Absolute baseline fallback: read the text using standard English engine rules
                voice_code = "en"
                using_fallback = True
                fallback_name = "English"

        # Generate Text-to-Speech (Audio)
        try:
            tts = gTTS(text=translated_text, lang=voice_code, slow=False)
            audio_filename = "translated_audio.mp3"
            tts.save(audio_filename)

            # If a fallback happened, notify the user gracefully in the output box


        except Exception as tts_error:
            translated_text += f"\n\n⚠️ Could not render audio clip: {str(tts_error)}"
            audio_filename = None

        return translated_text, audio_filename

    except Exception as e:
        return f"An error occurred during processing: {str(e)}", None


# Build the Gradio User Interface (UI)
with gr.Blocks() as demo:
    gr.Markdown("#🔴Language_Translator_Tool")

    # Sort choices alphabetically so users can navigate hundreds of options instantly
    source_choices = sorted(list(SOURCE_LANGUAGES.keys()))
    target_choices = sorted(list(ALL_WORLD_TEXT_LANGS.keys()))

    with gr.Row():
        # Left column: Input section
        with gr.Column():
            input_text = gr.Textbox(
                label="Enter Text",
                placeholder="Type or paste your text here...",
                lines=5
            )
            source_lang = gr.Dropdown(
                choices=source_choices,
                value='Auto Detect',
                label="Source Language"
            )
            target_lang = gr.Dropdown(
                choices=target_choices,
                value='Bhojpuri',  # Set to Bhojpuri by default so you can instantly verify the fix
                label="Target Language (Supports 240+ Languages)"
            )
            translate_btn = gr.Button("Translate & Generate Audio", variant="primary")

        # Right column: Output section
        with gr.Column():
            output_text = gr.Textbox(
                label="Translated Text",
                lines=7,
                interactive=False,
                show_copy_button=True
            )
            output_audio = gr.Audio(
                label="Audio Output (Universal Sound Stream)",
                type="filepath"
            )

    # Connect the button click event to the processing function
    translate_btn.click(
        fn=translate_and_tts,
        inputs=[input_text, source_lang, target_lang],
        outputs=[output_text, output_audio]
    )

# Launch the interface
demo.launch(debug=False)

Checking and installing required libraries. Please wait...
Libraries installed successfully!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a123e64ed648384fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
